In [1]:
#!wget -q -O detector.tflite -q https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite

In [6]:
from typing import Tuple, Union
import math
import cv2
import numpy as np

MARGIN = 10 
ROW_SIZE = 10 
FONT_SIZE = 1
FONT_THICKNESS = 1
TEXT_COLOR = (255, 0, 0)


def _normalized_to_pixel_coordinates(
    normalized_x: float, normalized_y: float, image_width: int,
    image_height: int) -> Union[None, Tuple[int, int]]:

  def is_valid_normalized_value(value: float) -> bool:
    return (value > 0 or math.isclose(0, value)) and (value < 1 or
                                                      math.isclose(1, value))

  if not (is_valid_normalized_value(normalized_x) and
          is_valid_normalized_value(normalized_y)):
    return None
  x_px = min(math.floor(normalized_x * image_width), image_width - 1)
  y_px = min(math.floor(normalized_y * image_height), image_height - 1)
  return x_px, y_px


def visualize(
    image,
    detection_result
) -> np.ndarray:
  annotated_image = image.copy()
  height, width, _ = image.shape

  for detection in detection_result.detections:
    bbox = detection.bounding_box
    start_point = bbox.origin_x, bbox.origin_y
    end_point = bbox.origin_x + bbox.width, bbox.origin_y + bbox.height
    cv2.rectangle(annotated_image, start_point, end_point, TEXT_COLOR, 3)

    for keypoint in detection.keypoints:
      keypoint_px = _normalized_to_pixel_coordinates(keypoint.x, keypoint.y,
                                                     width, height)
      color, thickness, radius = (0, 255, 0), 2, 2
      cv2.circle(annotated_image, keypoint_px, thickness, color, radius)

    category = detection.categories[0]
    category_name = category.category_name
    category_name = '' if category_name is None else category_name
    probability = round(category.score, 2)
    result_text = category_name + ' (' + str(probability) + ')'
    text_location = (MARGIN + bbox.origin_x,
                     MARGIN + ROW_SIZE + bbox.origin_y)
    cv2.putText(annotated_image, result_text, text_location, cv2.FONT_HERSHEY_PLAIN,
                FONT_SIZE, TEXT_COLOR, FONT_THICKNESS)

  return annotated_image

In [8]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mpython
from mediapipe.tasks.python import vision

base_options = mpython.BaseOptions(model_asset_path='detector.tflite')
options = vision.FaceDetectorOptions(base_options=base_options)
detector = vision.FaceDetector.create_from_options(options)

image = mp.Image.create_from_file('tom.png')

detection_result = detector.detect(image)

image_copy = np.copy(image.numpy_view())
cv2.imwrite('tom_copy.jpg',image_copy)
annotated_image = visualize(image_copy, detection_result)
cv2.imwrite('tom_annotated.jpg',annotated_image)
rgb_annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
cv2.imwrite('output_tom.jpg',rgb_annotated_image)
print('done!')

W0000 00:00:1787186595.966665    2365 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


done!


In [17]:
import face_recognition
import cv2
import numpy as np

In [18]:
tom_image = face_recognition.load_image_file("tom.png")
davis_image = face_recognition.load_image_file("red_carpet_davis.png")
atwell_image = face_recognition.load_image_file("red_carpet_atwell.png")
pegg_image = face_recognition.load_image_file("red_carpet_pegg.png")
mcquarrie_image = face_recognition.load_image_file("red_carpet_mcquarrie.png")

tom_encoding = face_recognition.face_encodings(tom_image)[0]
davis_encoding = face_recognition.face_encodings(davis_image)[0]
atwell_encoding = face_recognition.face_encodings(atwell_image)[0]
pegg_encoding = face_recognition.face_encodings(pegg_image)[0]
mcquarrie_encoding = face_recognition.face_encodings(mcquarrie_image)[0]

In [19]:
known_face_encodings = [tom_encoding,
                        davis_encoding,
                        pegg_encoding,
                        atwell_encoding,
                        mcquarrie_encoding
]

known_face_names = ['Tom_Cruise',
                    'Greg_Davis',
                    'Simon_Pegg',
                    'Hayley_Atwell',
                    'Chris_Mcquarrie'
]

In [20]:
image = cv2.imread("red_carpet.png")
rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [21]:
face_locations = face_recognition.face_locations(rgb_image)
face_encodings = face_recognition.face_encodings(
    rgb_image,
    face_locations
)

In [22]:
for face_encoding in face_encodings:

    matches = face_recognition.compare_faces(
        known_face_encodings,
        face_encoding
    )

In [23]:
num = 0
for (top, right, bottom, left), face_encoding in zip(
        face_locations,
        face_encodings):
    num += 1

    matches = face_recognition.compare_faces(
        known_face_encodings,
        face_encoding
    )

    name = f"Unknown {num}"

    if True in matches:
        first_match_index = matches.index(True)
        name = known_face_names[first_match_index]

    cv2.rectangle(
        image,
        (left, top),
        (right, bottom),
        (0, 255, 0),
        2
    )

    cv2.putText(
        image,
        name,
        (left, top - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

In [24]:
cv2.imwrite('output_carpet.jpg',image)

True